In [1]:
!pip install -q clearml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.8 MB/s eta 0:00:00


In [ ]:
%env CLEARML_WEB_HOST=
%env CLEARML_API_HOST=
%env CLEARML_FILES_HOST=
%env CLEARML_API_ACCESS_KEY=
%env CLEARML_API_SECRET_KEY=

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import torchvision.transforms as T
import torchvision.models as models
import torch.nn as nn
from tqdm.auto import tqdm
import os
import torch.optim as optim
import torch.nn.functional as F
from clearml import Task

DATA_PATH = Path('/kaggle/input/datasets/saveliymazovatov/occlusions/Datasets 2')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
from torch.utils.data import ConcatDataset

class CleanDataset(Dataset):
    def __init__(self, clean_dir_path, num_classes, transform=None, valid_extensions={'.jpg', '.jpeg', '.png'}):
        self.clean_dir_path = Path(clean_dir_path)
        self.num_classes = num_classes
        self.transform = transform
        self.samples = [
            p for p in self.clean_dir_path.rglob('*') 
            if p.is_file() and p.suffix.lower() in valid_extensions
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_path = self.samples[idx]
        image = Image.open(file_path).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)
        else:
            image = T.ToTensor()(image)

        target = torch.zeros(self.num_classes, dtype=torch.float32)
        return image, target

In [5]:
class OcclusionDataset(Dataset):
    def __init__(self, data_path, split='train', is_train=True, transform=None):
        self.data_path = Path(data_path)
        self.split = split
        self.is_train = is_train
        self.transform = transform
        self.samples = []

        all_folders = sorted([d.name for d in self.data_path.iterdir() if d.is_dir()])
        
        self.defect_classes = [c for c in all_folders if c.lower() not in {'clean', 'clear'}]
        self.defect_class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.defect_classes)}

        valid_extensions = {'.jpg', '.jpeg', '.png'}

        for folder_name in all_folders:
            cls_folder = self.data_path / folder_name
            for file_path in cls_folder.rglob('*'):
                if file_path.is_file() and file_path.suffix.lower() in valid_extensions:
                    if self.split in file_path.parts:
                        self.samples.append((file_path, folder_name))

    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        file_path, folder_name = self.samples[idx]
        image = Image.open(file_path).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)
        else:
            image = T.ToTensor()(image)

        target = torch.zeros(len(self.defect_classes), dtype=torch.float32)

        if folder_name.lower() not in {'clean', 'clear'}:
            cls_idx = self.defect_class_to_idx[folder_name]
            target[cls_idx] = 1.0

        return image, target

In [6]:
train_transform = T.Compose([
    T.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.98, 1.02)),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [7]:
train_defect_dataset = OcclusionDataset(data_path=DATA_PATH, split='train', is_train=True, transform=train_transform)
test_defect_dataset = OcclusionDataset(data_path=DATA_PATH, split='test', is_train=False, transform=test_transform)

num_classes = len(train_defect_dataset.defect_classes)

CLEAN_TRAIN_PATH = Path('/kaggle/input/datasets/saveliymazovatov/clean-dataset/clean/train')
CLEAN_TEST_PATH = Path('/kaggle/input/datasets/saveliymazovatov/clean-dataset/clean/test')

train_clean_dataset = CleanDataset(CLEAN_TRAIN_PATH, num_classes=num_classes, transform=train_transform)
test_clean_dataset = CleanDataset(CLEAN_TEST_PATH, num_classes=num_classes, transform=test_transform)

train_dataset = ConcatDataset([train_defect_dataset, train_clean_dataset])
test_dataset = ConcatDataset([test_defect_dataset, test_clean_dataset])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

In [8]:
class MLPHead(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int = 256, dropout_p: float = 0.3):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class OcclusionNet_V2(nn.Module):
    def __init__(self, num_classes: int = 7, hidden_dim: int = 256, dropout_p: float = 0.3):
        super().__init__()
        
        weights = models.EfficientNet_B3_Weights.DEFAULT
        self.backbone = models.efficientnet_b3(weights=weights)
        
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()

        self.heads = nn.ModuleList([
            MLPHead(in_features=in_features, hidden_dim=hidden_dim, dropout_p=dropout_p)
            for _ in range(num_classes)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        logits = [head(features) for head in self.heads]
        return torch.cat(logits, dim=1)

In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score
from clearml import Logger


def train_eval(model, optimizer, criterion, scheduler, epochs, train_loader, test_loader, 
               model_name, class_names, weights_path='./weights', logger=None, device=device):
    loses_history = []
    metric_history = []
    best_f1 = 0.0
    global_step = 0
    
    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(epochs):
        current_epoch = epoch + 1
        
        model.train()
        epoch_train_loss = 0.0
        
        for images, targets in tqdm(train_loader, desc=f'Epoch {current_epoch}/{epochs} [Train]', leave=False):
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, targets)

            scaler.scale(loss).backward()
            
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float('inf')).item()
            
            if logger is not None:
                logger.report_scalar(
                    title='Gradients',
                    series='L2 Gradient Norm',
                    value=grad_norm,
                    iteration=global_step
                )
            global_step += 1

            scaler.step(optimizer)
            scaler.update()

            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)
        loses_history.append(avg_train_loss)

        model.eval()
        epoch_val_loss = 0.0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for images, targets in tqdm(test_loader, desc=f'Epoch {current_epoch}/{epochs} [Val]', leave=False):
                images = images.to(device)
                targets = targets.to(device)
                
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    val_loss = F.binary_cross_entropy_with_logits(outputs, targets)
                
                epoch_val_loss += val_loss.item()
                
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                
                all_preds.append(preds.cpu())
                all_targets.append(targets.cpu())
        
        avg_val_loss = epoch_val_loss / len(test_loader)
        all_preds = torch.cat(all_preds, dim=0).numpy()
        all_targets = torch.cat(all_targets, dim=0).numpy()
        
        num_classes = all_targets.shape[1]
        class_f1_scores = []
        class_precisions = []
        class_recalls = []
        
        for i in range(num_classes):
            p = precision_score(all_targets[:, i], all_preds[:, i], average='binary', zero_division=0)
            r = recall_score(all_targets[:, i], all_preds[:, i], average='binary', zero_division=0)
            f = f1_score(all_targets[:, i], all_preds[:, i], average='binary', zero_division=0)
            
            class_precisions.append(p)
            class_recalls.append(r)
            class_f1_scores.append(f)
            
            if logger is not None:
                logger.report_scalar(title='Class Precision', series=class_names[i], value=p, iteration=current_epoch)
                logger.report_scalar(title='Class Recall', series=class_names[i], value=r, iteration=current_epoch)
                logger.report_scalar(title='Class F1-Score', series=class_names[i], value=f, iteration=current_epoch)
        
        clean_true = (all_targets.sum(axis=1) == 0).astype(int)
        clean_pred = (all_preds.sum(axis=1) == 0).astype(int)

        if clean_true.sum() > 0:
            p_clean = precision_score(clean_true, clean_pred, zero_division=0)
            r_clean = recall_score(clean_true, clean_pred, zero_division=0)
            f_clean = f1_score(clean_true, clean_pred, zero_division=0)

            if logger is not None:
                logger.report_scalar(title='Class Precision', series='Clean', value=p_clean, iteration=current_epoch)
                logger.report_scalar(title='Class Recall', series='Clean', value=r_clean, iteration=current_epoch)
                logger.report_scalar(title='Class F1-Score', series='Clean', value=f_clean, iteration=current_epoch)

        macro_precision = float(sum(class_precisions) / len(class_precisions))
        macro_recall = float(sum(class_recalls) / len(class_recalls))
        macro_f1 = float(sum(class_f1_scores) / len(class_f1_scores))
        metric_history.append(macro_f1)

        if logger is not None:
            logger.report_scalar(title='Loss', series='Train Loss', value=avg_train_loss, iteration=current_epoch)
            logger.report_scalar(title='Loss', series='Val Loss', value=avg_val_loss, iteration=current_epoch)
            
            logger.report_scalar(title='Macro Metrics', series='Macro Precision', value=macro_precision, iteration=current_epoch)
            logger.report_scalar(title='Macro Metrics', series='Macro Recall', value=macro_recall, iteration=current_epoch)
            logger.report_scalar(title='Macro Metrics', series='Macro F1', value=macro_f1, iteration=current_epoch)
            
            current_lr = optimizer.param_groups[0]['lr']
            logger.report_scalar(title='Learning Rate', series='LR', value=current_lr, iteration=current_epoch)

        if scheduler is not None:
            scheduler.step()

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            os.makedirs(weights_path, exist_ok=True)
            best_weight_file = os.path.join(weights_path, f'{model_name}_best.pth')
            torch.save(model.state_dict(), best_weight_file)

        print(f"\nEpoch {current_epoch}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"Macro P: {macro_precision:.4f} | Macro R: {macro_recall:.4f} | Macro F1: {macro_f1:.4f}")
        print("Detailed Metrics by Class:")
        for idx, name in enumerate(class_names):
            print(f"  {name:<16} -> P: {class_precisions[idx]:.4f} | R: {class_recalls[idx]:.4f} | F1: {class_f1_scores[idx]:.4f}")
        if clean_true.sum() > 0:
            print(f"  {'Clean':<16} -> P: {p_clean:.4f} | R: {r_clean:.4f} | F1: {f_clean:.4f}")

    return loses_history, metric_history

In [10]:
EPOCHS = 20

weights_path = './weights'

os.makedirs(weights_path, exist_ok=True)

In [ ]:
task = Task.init(
    project_name='OcclusionNet',
    task_name='Misha_20_epochs_OcclusionNet_V2_EfficientNetB3_BCE',
    tags=['BCEWithLogitsLoss', 'MultiHead', 'EfficientNetB3', 'warmup']
)

task.set_comment("Обучение с backbone EfficientNet-B3 с 7 MLP-головами")

params = {
    'epochs': EPOCHS,
    'batch_size': 16,
    'lr_backbone': 1e-5,
    'lr_heads': 2e-4,
    'weight_decay': 1e-3,
    'backbone': 'EfficientNet_B3',
}
task.connect(params)

logger = task.get_logger()

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

WARMUP_EPOCHS = 3
COSINE_EPOCHS = EPOCHS - WARMUP_EPOCHS

defect_classes = train_defect_dataset.defect_classes

model = OcclusionNet_V2(num_classes=len(defect_classes), hidden_dim=256, dropout_p=0.3).to(device)
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-5, 'weight_decay': 1e-3},
    {'params': model.heads.parameters(), 'lr': 2e-4, 'weight_decay': 1e-3}
])

warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimizer, 
    start_factor=0.1,
    end_factor=1.0, 
    total_iters=WARMUP_EPOCHS
)

cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=COSINE_EPOCHS, 
    eta_min=1e-6
)

scheduler = optim.lr_scheduler.SequentialLR(
    optimizer, 
    schedulers=[warmup_scheduler, cosine_scheduler], 
    milestones=[WARMUP_EPOCHS]
)

loses_history, metric_history = train_eval(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=EPOCHS,
    train_loader=train_loader,
    test_loader=test_loader,
    model_name='OcclusionNet_V2',
    class_names=defect_classes,
    weights_path=weights_path,
    logger=logger,
    device=device
)

best_model_file = os.path.join(weights_path, 'OcclusionNet_V2_best.pth')
model.load_state_dict(torch.load(best_model_file))
model.eval()

all_probs = []
all_targets = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc='Evaluating Best Model'):
        images = images.to(device)
        targets = targets.to(device)
        
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            
        all_probs.append(probs.cpu())
        all_targets.append(targets.cpu())

all_probs = torch.cat(all_probs, dim=0).numpy()
all_targets = torch.cat(all_targets, dim=0).numpy()

class_names = defect_classes
num_classes = len(class_names)

best_thresholds = []
candidate_thresholds = np.arange(0.1, 0.9, 0.02)

for idx in range(num_classes):
    best_th = 0.5
    best_th_f1 = -1.0
    for th in candidate_thresholds:
        preds_th = (all_probs[:, idx] >= th).astype(int)
        score = f1_score(all_targets[:, idx], preds_th, average='binary', zero_division=0)
        if score > best_th_f1:
            best_th_f1 = score
            best_th = th
    best_thresholds.append(best_th)

def compute_metrics(probs, targets, thresholds, class_names):
    if isinstance(thresholds, (float, int)):
        thresholds = [thresholds] * len(class_names)
        
    preds = np.zeros_like(probs)
    for idx, th in enumerate(thresholds):
        preds[:, idx] = (probs[:, idx] >= th).astype(float)
        
    report_data = []
    for idx, name in enumerate(class_names):
        p = precision_score(targets[:, idx], preds[:, idx], average='binary', zero_division=0)
        r = recall_score(targets[:, idx], preds[:, idx], average='binary', zero_division=0)
        f = f1_score(targets[:, idx], preds[:, idx], average='binary', zero_division=0)
        report_data.append({
            'Class': name,
            'Threshold': round(float(thresholds[idx]), 2),
            'Precision': round(float(p), 4),
            'Recall': round(float(r), 4),
            'F1-Score': round(float(f), 4),
            'Support': int(targets[:, idx].sum())
        })
        
    clean_true = (targets.sum(axis=1) == 0).astype(int)
    clean_pred = (preds.sum(axis=1) == 0).astype(int)
    if clean_true.sum() > 0:
        p_clean = precision_score(clean_true, clean_pred, zero_division=0)
        r_clean = recall_score(clean_true, clean_pred, zero_division=0)
        f_clean = f1_score(clean_true, clean_pred, zero_division=0)
        report_data.append({
            'Class': 'Clean',
            'Threshold': '-',
            'Precision': round(float(p_clean), 4),
            'Recall': round(float(r_clean), 4),
            'F1-Score': round(float(f_clean), 4),
            'Support': int(clean_true.sum())
        })
        
    defect_rows = [d for d in report_data if d['Class'] != 'Clean']
    macro_p_defects = float(np.mean([d['Precision'] for d in defect_rows]))
    macro_r_defects = float(np.mean([d['Recall'] for d in defect_rows]))
    macro_f1_defects = float(np.mean([d['F1-Score'] for d in defect_rows]))
    
    macro_p_all = float(np.mean([d['Precision'] for d in report_data]))
    macro_r_all = float(np.mean([d['Recall'] for d in report_data]))
    macro_f1_all = float(np.mean([d['F1-Score'] for d in report_data]))
    
    table_rows = list(report_data)
    table_rows.append({
        'Class': 'Macro (Defects Only)', 'Threshold': '-',
        'Precision': round(macro_p_defects, 4), 'Recall': round(macro_r_defects, 4),
        'F1-Score': round(macro_f1_defects, 4), 'Support': int(sum([d['Support'] for d in defect_rows]))
    })
    table_rows.append({
        'Class': 'Macro (With Clean)', 'Threshold': '-',
        'Precision': round(macro_p_all, 4), 'Recall': round(macro_r_all, 4),
        'F1-Score': round(macro_f1_all, 4), 'Support': int(sum([d['Support'] for d in report_data]))
    })
    return pd.DataFrame(table_rows), report_data

df_05, report_05 = compute_metrics(all_probs, all_targets, 0.5, class_names)
df_tuned, report_tuned = compute_metrics(all_probs, all_targets, best_thresholds, class_names)

thresholds_dict = {name: round(float(th), 2) for name, th in zip(class_names, best_thresholds)}

summary_metrics = {
    'Best_Thresholds': thresholds_dict,
    'Tuned_Thresholds': {
        'Macro_Without_Clean': df_tuned[df_tuned['Class'] == 'Macro (Defects Only)'].to_dict(orient='records')[0],
        'Macro_With_Clean': df_tuned[df_tuned['Class'] == 'Macro (With Clean)'].to_dict(orient='records')[0],
        'Classes': report_tuned
    },
    'Default_0.5_Threshold': {
        'Macro_Without_Clean': df_05[df_05['Class'] == 'Macro (Defects Only)'].to_dict(orient='records')[0],
        'Macro_With_Clean': df_05[df_05['Class'] == 'Macro (With Clean)'].to_dict(orient='records')[0],
        'Classes': report_05
    }
}

thresholds_json_path = os.path.join(weights_path, 'best_thresholds.json')
with open(thresholds_json_path, 'w') as f:
    json.dump(thresholds_dict, f, indent=4)

metrics_json_path = os.path.join(weights_path, 'best_metrics.json')
with open(metrics_json_path, 'w') as f:
    json.dump(summary_metrics, f, indent=4)

metrics_csv_path_05 = os.path.join(weights_path, 'metrics_th_0.5.csv')
metrics_csv_path_tuned = os.path.join(weights_path, 'metrics_th_tuned.csv')
df_05.to_csv(metrics_csv_path_05, index=False)
df_tuned.to_csv(metrics_csv_path_tuned, index=False)

task.upload_artifact(name='best_weights', artifact_object=best_model_file)
task.upload_artifact(name='best_thresholds_json', artifact_object=thresholds_json_path)
task.upload_artifact(name='best_metrics_json', artifact_object=metrics_json_path)
task.upload_artifact(name='metrics_csv_0.5', artifact_object=metrics_csv_path_05)
task.upload_artifact(name='metrics_csv_tuned', artifact_object=metrics_csv_path_tuned)

if logger is not None:
    logger.report_table(title='Evaluation', series='Threshold_0.5', table_plot=df_05)
    logger.report_table(title='Evaluation', series='Threshold_Tuned', table_plot=df_tuned)

print("Результаты со стандартным порогом 0.5")
print(df_05.to_string(index=False))
print("\nРезультаты с оптимизированными порогами")
print(df_tuned.to_string(index=False))

task.close()